In [13]:
from google.colab import drive
from pathlib import Path

# Use the existing Drive mount if it is already mounted.
DRIVE = Path("/content/drive/MyDrive")
PROJECT = DRIVE / "Ocular_Project"

print("Project exists:", PROJECT.exists())
print("Project contents:")
if PROJECT.exists():
    for p in PROJECT.iterdir():
        print("DIR " if p.is_dir() else "FILE", p.name)

print("\nIITD archive:", (PROJECT / "datasets/IITD_database (1).7z").exists())
print("IITD extracted folder:", (PROJECT / "datasets/IITD_database").exists())

print("\nSearching project results for Notebook 01 metadata:")
for p in (PROJECT / "results").rglob("IITD_dataset_metadata.csv"):
    print("FOUND:", p)

Project exists: True
Project contents:
DIR  results

IITD archive: False
IITD extracted folder: False

Searching project results for Notebook 01 metadata:


In [12]:

from google.colab import drive
from pathlib import Path

# Inspect the existing directory without deleting anything
old_path = Path("/content/drive")
print("Existing /content/drive contents:")
if old_path.exists():
    for p in list(old_path.iterdir())[:10]:
        print(" -", p.name)
else:
    print("Directory does not exist")

# Mount Google Drive at a NEW, clean location
drive.mount("/content/gdrive")

# Verify access to your project
PROJECT = Path("/content/gdrive/MyDrive/Ocular_Project")

print("\nProject exists:", PROJECT.exists())
if PROJECT.exists():
    print("Project contents:")
    for p in PROJECT.iterdir():
        print(("DIR  " if p.is_dir() else "FILE "), p.name)

print("\nIITD archive exists:",
      (PROJECT / "datasets/IITD_database (1).7z").exists())


Existing /content/drive contents:
 - MyDrive
Mounted at /content/gdrive

Project exists: True
Project contents:
DIR   datasets
DIR   models
DIR   generated_images
DIR   results

IITD archive exists: True


In [14]:

from pathlib import Path

PROJECT = Path("/content/gdrive/MyDrive/Ocular_Project")
DATASETS = PROJECT / "datasets"
RESULTS = PROJECT / "results"

print("=== IITD dataset folders ===")
for p in DATASETS.iterdir():
    print(("DIR  " if p.is_dir() else "FILE "), p.name)

print("\n=== Notebook 01 metadata files ===")
metadata_files = list(RESULTS.rglob("IITD_dataset_metadata.csv"))

if metadata_files:
    for p in metadata_files:
        print("FOUND:", p)
else:
    print("Metadata CSV not found. Notebook 01 outputs may be in another location.")


=== IITD dataset folders ===
FILE  archive.zip
DIR   CASIA-Iris-Thousand
FILE  IITD_database (1).7z
DIR   IITD_database

=== Notebook 01 metadata files ===
FOUND: /content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook01_dataset_analysis/IITD_dataset_metadata.csv


In [15]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive/MyDrive/Ocular_Project")
RESULTS = PROJECT / "results"
DATASETS = PROJECT / "datasets"

META_PATH = (
    RESULTS / "IITD/notebook01_dataset_analysis/"
    "IITD_dataset_metadata.csv"
)

df = pd.read_csv(META_PATH)

print("Metadata loaded:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 3 rows:")
display(df.head(3))

# Show extracted dataset location
for p in DATASETS.rglob("01_L.bmp"):
    print("\nExample image:", p)
    print("Image exists:", p.exists())
    break

# Inspect path-related metadata columns
path_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in ["path", "file", "image"])
]
print("\nPath-related columns:", path_cols)

for c in path_cols:
    print(f"\n{c} examples:")
    print(df[c].head(3).tolist())


Metadata loaded: (3270, 13)

Columns:
['image_path', 'relative_path', 'filename', 'subject', 'eye', 'capture_id', 'extension', 'valid_image', 'width', 'height', 'mode', 'format', 'file_size_bytes']

First 3 rows:


,image_path,relative_path,filename,subject,eye,capture_id,extension,valid_image,width,height,mode,format,file_size_bytes
0,/content/drive/MyDrive/Ocular_Project/datasets...,001/01_L.bmp,01_L.bmp,001,L,1,.bmp,True,320,240,RGB,BMP,230454
1,/content/drive/MyDrive/Ocular_Project/datasets...,001/02_L.bmp,02_L.bmp,001,L,2,.bmp,True,320,240,RGB,BMP,230454
2,/content/drive/MyDrive/Ocular_Project/datasets...,001/03_L.bmp,03_L.bmp,001,L,3,.bmp,True,320,240,RGB,BMP,230454



Example image: /content/gdrive/MyDrive/Ocular_Project/datasets/IITD_database/IITD_database/207/01_L.bmp
Image exists: True

Path-related columns: ['image_path', 'relative_path', 'filename', 'valid_image', 'file_size_bytes']

image_path examples:
['/content/drive/MyDrive/Ocular_Project/datasets/IITD_database/IITD_database/001/01_L.bmp', '/content/drive/MyDrive/Ocular_Project/datasets/IITD_database/IITD_database/001/02_L.bmp', '/content/drive/MyDrive/Ocular_Project/datasets/IITD_database/IITD_database/001/03_L.bmp']

relative_path examples:
['001/01_L.bmp', '001/02_L.bmp', '001/03_L.bmp']

filename examples:
['01_L.bmp', '02_L.bmp', '03_L.bmp']

valid_image examples:
[True, True, True]

file_size_bytes examples:
[230454, 230454, 230454]


In [16]:

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm

PROJECT = Path("/content/gdrive/MyDrive/Ocular_Project")
DATASET_ROOT = PROJECT / "datasets/IITD_database/IITD_database"
OUT_DIR = PROJECT / "results/IITD/notebook02_preprocessing"
IMAGE_OUT = OUT_DIR / "images_224"

OUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_OUT.mkdir(parents=True, exist_ok=True)

META_PATH = (
    PROJECT / "results/IITD/notebook01_dataset_analysis/"
    "IITD_dataset_metadata.csv"
)
df = pd.read_csv(META_PATH)

# Resolve the old image paths using relative_path
df["resolved_path"] = df["relative_path"].apply(
    lambda p: str(DATASET_ROOT / str(p))
)
df["path_exists"] = df["resolved_path"].apply(
    lambda p: Path(p).is_file()
)

print("Metadata rows:", len(df))
print("Images found:", int(df["path_exists"].sum()))
print("Images missing:", int((~df["path_exists"]).sum()))

# Stop if the root/path assumption is wrong
if not df["path_exists"].all():
    display(df.loc[~df["path_exists"], ["relative_path", "resolved_path"]].head(10))
    raise FileNotFoundError(
        "Some image paths are missing. Check DATASET_ROOT before continuing."
    )

records = []
errors = []

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Preprocessing"):
    src = Path(row.resolved_path)
    # Keep the subject folders, avoiding filename collisions
    dest = IMAGE_OUT / Path(row.relative_path).with_suffix(".png")
    dest.parent.mkdir(parents=True, exist_ok=True)

    try:
        with Image.open(src) as im:
            im = ImageOps.grayscale(im)
            im = im.resize((224, 224), Image.Resampling.LANCZOS)
            im.save(dest, format="PNG")

        records.append({
            "original_relative_path": row.relative_path,
            "processed_path": str(dest),
            "subject": row.subject,
            "eye": row.eye,
            "capture_id": row.capture_id,
            "width": 224,
            "height": 224,
            "mode": "L"
        })
    except Exception as e:
        errors.append({
            "relative_path": row.relative_path,
            "error": str(e)
        })

processed_df = pd.DataFrame(records)
processed_csv = OUT_DIR / "IITD_preprocessed_metadata.csv"
processed_df.to_csv(processed_csv, index=False)

errors_csv = OUT_DIR / "IITD_preprocessing_errors.csv"
pd.DataFrame(errors).to_csv(errors_csv, index=False)

print("\nPreprocessing complete")
print("Successfully processed:", len(processed_df))
print("Errors:", len(errors))
print("Metadata saved:", processed_csv)
print("Errors saved:", errors_csv)

display(processed_df.head())


Metadata rows: 3270
Images found: 3270
Images missing: 0


Preprocessing:   0%|          | 0/3270 [00:00<?, ?it/s]


Preprocessing complete
Successfully processed: 3270
Errors: 0
Metadata saved: /content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook02_preprocessing/IITD_preprocessed_metadata.csv
Errors saved: /content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook02_preprocessing/IITD_preprocessing_errors.csv


,original_relative_path,processed_path,subject,eye,capture_id,width,height,mode
0,001/01_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,1,224,224,L
1,001/02_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,2,224,224,L
2,001/03_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,3,224,224,L
3,001/04_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,4,224,224,L
4,001/05_L.bmp,/content/gdrive/MyDrive/Ocular_Project/results...,001,L,5,224,224,L


## Conclusion

In this notebook, the IIT Delhi Iris Database was preprocessed to prepare the images for subsequent ocular recognition experiments. The dataset metadata generated in Notebook 01 was loaded, and all image paths were resolved using their relative paths to accommodate the current Google Drive mount.

A total of **3,270 images** were processed successfully, with **zero preprocessing errors**. Each image was converted to grayscale and resized to **224 × 224 pixels** to provide a consistent input format for subsequent experiments.

The preprocessed images were saved as PNG files while preserving the subject-wise directory structure. The corresponding metadata was exported to `IITD_preprocessed_metadata.csv`, and preprocessing errors were recorded in `IITD_preprocessing_errors.csv`.

### Key Outcomes

- **Total images processed:** 3,270
- **Successfully processed:** 3,270
- **Preprocessing errors:** 0
- **Output dimensions:** 224 × 224 pixels
- **Output image format:** Grayscale PNG
- **Main output:** Preprocessed images and their metadata

### Next Steps

The preprocessed dataset will be used in Notebook 03 for image quality assessment, including brightness, contrast, sharpness, and an exploratory noise estimate. These measurements will help characterize the dataset before training the baseline ocular recognition model.

**Note:** The original dataset contains images with different source dimensions and formats. Converting all images to a common grayscale resolution ensures consistency for the planned experiments, but it may remove some information from the original images. Therefore, the effects of preprocessing should be considered when interpreting subsequent recognition results.